# SQL Query Chain

자연어 질문을 **SQL 쿼리로 변환 → DB 에서 실행 → 결과를 자연어 답변으로 정리**하는 체인을 만듭니다.

```
질문 ──▶ [write_query] ──▶ SQL ──▶ [execute_query] ──▶ 결과 ──▶ [answer] ──▶ 답변
       (LLM이 SQL 생성)          (DB에서 실행)              (LLM이 답변 작성)
```

| 단계 | 구성 요소 | 역할 |
|---|---|---|
| 1 | `create_sql_query_chain` | DB 스키마를 보고 질문에 맞는 SQL 을 생성 |
| 2 | `QuerySQLDataBaseTool` | 생성된 SQL 을 DB 에 실행하고 결과를 문자열로 반환 |
| 3 | `answer_prompt \| llm` | 질문 + SQL + 결과를 보고 최종 답변 작성 |

**실습 DB**: `data/finance.db` (SQLite) - `accounts`, `customers`, `transactions` 테이블

## 1. 환경 설정 & DB 연결

`SQLDatabase` 는 SQLAlchemy 기반의 DB 래퍼입니다. URI 만 바꾸면 SQLite 외에 MySQL, PostgreSQL 등에도 연결할 수 있습니다.
- `db.dialect`: SQL 방언(sqlite, mysql, postgresql ...). LLM 이 **해당 DB 문법**에 맞는 SQL 을 쓰도록 프롬프트에 전달됩니다.
- `db.get_usable_table_names()`: 사용 가능한 테이블 목록

In [49]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain  # LangChain 1.x 에서 langchain_classic 으로 이동한 레거시 체인
from langchain_community.utilities import SQLDatabase
from langchain_teddynote import logging
from dotenv import load_dotenv

# .env 에 저장된 API KEY 로드
load_dotenv()

# LangSmith 추적 (프로젝트 이름: SQL)
logging.langsmith("SQL")

# SQLite DB 연결 (노트북 위치 기준 상위 폴더의 data/finance.db)
db = SQLDatabase.from_uri("sqlite:///../data/finance.db")

# DB 종류(방언) 확인
print(db.dialect)

# 조회 가능한 테이블 목록
print(db.get_usable_table_names())

LangSmith 추적을 시작합니다.
[프로젝트명]
SQL
sqlite
['accounts', 'customers', 'transactions']


> 💡 LLM 이 실제로 보는 정보가 궁금하다면 `print(db.get_table_info())` 를 실행해 보세요.
> 각 테이블의 **스키마(CREATE TABLE 문) + 예시 데이터 3행**이 출력되며, 이것이 SQL 생성 프롬프트에 들어갑니다.

## 2. SQL 생성 체인 (`create_sql_query_chain`)

`create_sql_query_chain(llm, db)` 는 내부적으로
1. `db.get_table_info()` 로 **테이블 스키마**를 가져오고,
2. `db.dialect` 에 맞는 프롬프트에 **스키마 + 질문**을 넣어,
3. LLM 이 **SQL 문자열만** 생성하도록 합니다. (실행은 하지 않습니다)

입력은 `{"question": "..."}` 형식의 딕셔너리입니다.

In [50]:
# temperature=0: SQL 은 정확해야 하므로 무작위성을 없앱니다.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 질문 → SQL 문자열을 만드는 체인
chain = create_sql_query_chain(llm, db)

In [51]:
# 자연어 질문을 SQL 로 변환 (아직 DB 에서 실행하지는 않음)
generated_sql_query = chain.invoke({"question": "고객의 이름을 나열하세요"})

생성된 SQL 을 확인합니다. `__repr__()` 로 출력하면 줄바꿈(`\n`) 같은 숨은 문자까지 보입니다.

> ⚠️ 모델에 따라 `'SQLQuery: SELECT ...'` 처럼 **앞에 라벨이 붙어 나오는 경우**가 있습니다.
> 그대로 실행하면 `near "SQLQuery": syntax error` 가 나므로, 이때는 라벨을 제거하는 후처리가 필요합니다.

In [52]:
# 생성된 SQL 을 있는 그대로(숨은 문자 포함) 출력
print(generated_sql_query.__repr__())

'SELECT "name" \nFROM customers'


## 3. SQL 실행 (`QuerySQLDataBaseTool`)

생성된 SQL 을 실제 DB 에 실행하는 **도구(Tool)** 입니다.
- 입력: `{"query": "SELECT ..."}`
- 출력: 결과 행들을 **문자열**로 반환 (예: `"[('테디',), ('폴',)]"`)
- SQL 에 오류가 있어도 예외를 던지지 않고 `"Error: ..."` 문자열을 반환합니다.

> ⚠️ LLM 이 만든 SQL 을 그대로 실행하므로, 실제 서비스에서는 **읽기 전용 계정**으로 연결해 `DELETE`/`DROP` 같은 위험한 쿼리를 막아야 합니다.
>
> 참고: `QuerySQLDataBaseTool` 은 deprecated 되었고 새 이름은 `from langchain_community.tools import QuerySQLDatabaseTool` 입니다. (사용법 동일)

In [53]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

# DB 에 SQL 을 실행하는 도구
execute_query = QuerySQLDataBaseTool(db=db)

# 앞에서 생성한 SQL 을 실행
execute_query.invoke({"query": generated_sql_query})

"[('테디',), ('폴',), ('셜리',), ('민수',), ('지영',), ('은정',)]"

### SQL 생성 + 실행을 하나의 체인으로

`|` 로 연결하면 **write_query 의 출력(SQL 문자열)이 그대로 execute_query 의 입력**이 됩니다.
(Tool 은 문자열 입력 하나만 받으면 자동으로 `query` 인자로 넣어 줍니다.)

결과는 DB 의 **원시 데이터**(튜플 리스트 문자열)입니다.

In [54]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

execute_query = QuerySQLDataBaseTool(db=db)   # SQL 실행 도구
write_query = create_sql_query_chain(llm, db)  # 질문 → SQL 생성 체인

# 질문 → SQL 생성 → DB 실행
chain = write_query | execute_query

In [55]:
# 결과는 자연어가 아닌 DB 조회 결과(문자열) 그대로 나옵니다.
chain.invoke({"question": "테디의 이메일을 조회하세요."})

"[('teddy@example.com',)]"

## 4. 결과를 자연어 답변으로 만들기

DB 결과(`[('teddy@example.com',)]`)는 사람이 읽기 불편하므로, LLM 에게 **질문 + SQL + 결과**를 함께 주고 답변을 작성하게 합니다.

In [56]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 질문(question), 생성된 SQL(query), 실행 결과(result)를 받아 답변을 작성하는 프롬프트
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

    Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer: """
)

> ⚠️ **아래 셀을 반드시 실행한 뒤** 다음 셀로 넘어가세요.
> 건너뛰면 `answer` 변수가 체인이 아니어서 `TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given` 이 발생합니다.

In [ ]:
# 답변 생성 체인: 프롬프트 → LLM → 문자열
answer = answer_prompt | llm | StrOutputParser()

### 전체 체인 조립

`RunnablePassthrough.assign(키=체인)` 은 **입력 딕셔너리를 유지한 채 새 키를 추가**합니다.
단계마다 딕셔너리가 아래처럼 점점 채워집니다.

```python
{"question": "..."}                                       # 입력
→ {"question": "...", "query": "SELECT ..."}              # .assign(query=write_query)
→ {"question": "...", "query": "SELECT ...", "result": "[...]"}  # .assign(result=...)
→ "테디의 거래 합계는 ... 입니다."                          # | answer
```

- `itemgetter("query")`: 딕셔너리에서 `query` 값(SQL 문자열)만 꺼내 `execute_query` 에 전달합니다.

In [ ]:
chain = (
    # 1) question 으로 SQL 을 생성해 query 키에 추가
    RunnablePassthrough.assign(query=write_query).assign(
        # 2) query 를 꺼내 DB 에 실행하고, 결과를 result 키에 추가
        result=itemgetter("query") | execute_query
    )
    # 3) question + query + result 로 최종 답변 생성
    | answer
)

In [61]:
# 질문 → SQL 생성 → 실행 → 자연어 답변
chain.invoke({"question": "테디의 transaction의 합계를 구하세요."})

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given

## 정리 & 참고

- `create_sql_query_chain` 은 **레거시 체인**(`langchain_classic`)입니다. 요즘은 LangGraph 또는 `create_agent` + `SQLDatabaseToolkit` 으로
  **SQL 생성 → 실행 → 오류 시 수정 재시도**까지 에이전트가 스스로 하도록 구성하는 방식을 권장합니다.
- 결과가 이상하다면 LangSmith(프로젝트 `SQL`)에서 **LLM 이 생성한 SQL** 을 먼저 확인하세요. 대부분의 오답은 SQL 이 질문을 잘못 해석한 데서 생깁니다.